# Tutorial 06 - Stereo

Agenda:

- Calibración Stereo
- Rectificación Stereo
- Mapas de Disparidad
- Profundidad y nube de puntos 3D densa

# Setup

Este tutorial se puede ejecutar local en Jupyter lab (o utilizar Google Colab, en casi su totalidad).

**La herramienta de calibración stéreo requiere utilizar la GUI del sistema operativo por lo no va a funcionar en Google Colab**.


## En Google Colab 
Este tutorial se provee junto con archivos de recursos dentro de un archivo ".zip".
En caso de ejecutar en Google Colab hay que:

1. Descomprimir el zip en algún lado
2. Subir el contenido del zip a Google Drive en alguna carpeta (por ejemplo `udesa/I308/tutoriales/tutorial_X`)
3. Abrir este notebook .ipynb 

In [ ]:
import os
import sys

# TODO: establecer el path en caso de trabajar con Colab
DRIVE_DIR = "udesa/I308/tutoriales/tutorial_06"

# detecta si estamos corriendo en Google Colab
try:
  from google.colab import drive
  COLAB = True
except:
  COLAB = False

if COLAB:
    # monta Google Drive
    drive.mount('/content/drive')

    base_path = "/content/drive/MyDrive/"
    path = os.path.join(base_path, DRIVE_DIR)
    
    %cd {path}
    sys.path.append(path)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# instalamos el paquete de utilidades
!pip install -qq git+https://github.com/udesa-vision/i308-utils.git

from i308_utils import imshow, show_images

In [ ]:
from setup import download_stereo_data

download_stereo_data()

# Calibración Estéreo

Hoy calibraremos una cámara estéreo [ELP-USB3D1080P02-H120](https://www.webcamerausb.com/elp-4mp-sync-usb-camera-with-dual-lens-wide-angle-120dergee-micro-distortion-binocular-mini-uvc-fhd-usb20-webcam-board-face-recognition-camera-compatible-raspberry-pi-linux-android-windows-p-478.html)


<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_05/ELP-USB3D1080P02.jpg?raw=true" width="50%"/>


## Notas

- esta cámara **no tiene auto-foco**
- la camara quedará correctamente orientada cuando el **conector USB apunta para arriba**.
  
- esta cámara sincroniza ambas capturas (izquierda y derecha) y genera un único frame concatenando ambas imágenes horizontalmente [L | R]. En python se pueden separar usando slicing de numpy.

- permite utilizar distintas resoluciones (idealmente trabajemos con 3840x1080, de modo tal que cada imagen capturada queda en Full HD (1920x1080) píxeles).


## Calibración Stereo en Notebook 

In [ ]:
# Definición del checkerboard

checkerboard = (10, 7)
square_size_mm = 24.2


In [ ]:
from calib import board_points

# creamos los puntos del mundo del objeto checkerboard
checkerboard_world_points_mm = board_points(checkerboard) * square_size_mm


In [ ]:
import glob, os

directory = os.path.join("stereo_data", "calib")
left_files_pattern = "*left*.jpg"
right_files_pattern = "*right*.jpg"

def numeric_sort(file_name):
    return int(file_name.split("_")[-1].split(".")[0])

left_file_names = sorted(
    glob.glob(
        os.path.join(directory, left_files_pattern)
    ),
    key=numeric_sort
)

right_file_names = sorted(
    glob.glob(
        os.path.join(directory, right_files_pattern)
    ),
    key=numeric_sort
)

num_left = len(left_file_names)
num_right = len(right_file_names)

if  num_left != num_right:
    raise Exception(f"the number of files (left {num_left} / right{num_right}) doesn't match")

In [ ]:
import cv2

image_size = None
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

world_points = []
left_images = []
right_images = []
left_images_points = []
right_images_points = []

for left_file_name, right_file_name in zip(
    left_file_names, right_file_names
):

    print("processing", left_file_name, right_file_name)

    # read left and right images
    left_image = cv2.imread(left_file_name, cv2.IMREAD_GRAYSCALE)
    right_image = cv2.imread(right_file_name, cv2.IMREAD_GRAYSCALE)

    # get the images sizes
    left_size = (left_image.shape[1], left_image.shape[0])
    right_size = (right_image.shape[1], right_image.shape[0])

    # checks that images sizes match
    if left_size != right_size:
        raise Exception(f"left and right images sizes differ: left {left_size} / right {right_size}")
        
    if image_size is None:
        # remembers the images size
        image_size = left_size
    else:
        if image_size != left_size:
            raise Exception(f"there are images with different sizes: {image_size} vs {left_size}")

    # finds the checkerboard in each image
    left_found, left_corners = cv2.findChessboardCorners(left_image, checkerboard)
    right_found, right_corners = cv2.findChessboardCorners(right_image, checkerboard)

    if not left_found or not right_found:
        print("warning, checkerboard was not found")
        continue

    # checkerboard was found in both images.

    # let's improve the found corners
    corners_left = cv2.cornerSubPix(left_image, left_corners, (7, 7), (-1,-1), criteria)
    corners_right = cv2.cornerSubPix(right_image, right_corners, (7, 7), (-1,-1), criteria)

    # acumulo las imagenes
    left_images.append(left_image)
    right_images.append(right_image)

    # acumulo los corners detectados
    left_images_points.append(left_corners)
    right_images_points.append(right_corners)

    # acumulo los puntos del mundo
    world_points.append(checkerboard_world_points_mm)



In [ ]:
show_images([
    left_images[14], right_images[14]
], ['left', 'right'])

In [ ]:
# grafiquemos algunos checkerboards
# y sus detecciones de corners
#  para verificar que lo que encontró está ok:

import calib
some_images_indices = [5, 14]

draw_settings = {
    "corner_radius": 10,
    "corner_thickness": 5,
    "line_thickness": 4
}
for i in some_images_indices:

    left_img, right_img = left_images[i], right_images[i]
    left_corners, right_corners = left_images_points[i], right_images_points[i]

    show_left = cv2.cvtColor(left_img, cv2.COLOR_GRAY2BGR)
    show_right = cv2.cvtColor(right_img, cv2.COLOR_GRAY2BGR)


    calib.draw_checkerboard(show_left, checkerboard, left_corners, True, **draw_settings)
    calib.draw_checkerboard(show_right, checkerboard, right_corners, True, **draw_settings)
    

    show_images([show_left, show_right], ["left", "right"])
    


## Calibración Stereo con OpenCV

OpenCV nos provee la función [`cv2.stereoCalibrate`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga9d2539c1ebcda647487a616bdf0fc716).
que realiza la **calibración estéreo** de un sistema de dos cámaras. 

*"básicamente encuentra la pose relativa entre ambas cámaras"*

Esta función estima:

- Los **parámetros intrínsecos** de cada cámara (`K1`, `distCoeffs1`, `K2`, `distCoeffs2`).
- Los **parámetros extrínsecos** rotación y traslación relativa entre ambas cámaras (`R` y `t` que transforman puntos 3d del sistema de la cámara 1 (izquierda) al sistema de la cámara 2 (derecha)).
- Las **matrices esencial** y **fundamental**, (`E` y `F`) que describen la geometría epipolar entre las cámaras.

**parámetros de entrada**

- `objectPoints`: Puntos 3D conocidos del patrón (como los de un checkerboard), iguales para ambas cámaras.
- `imagePoints1`, `imagePoints2`: Puntos detectados en las imágenes 2D de cada cámara.
- `cameraMatrix1`, `cameraMatrix2`: Matrices intrínsecas de ambas cámaras.
- `distCoeffs1`, `distCoeffs2`: Coeficientes de distorsión de cada cámara.
- `imageSize`: Tamaño de las imágenes usadas.
- Opcionalmente, criterios de terminación (`criteria`) y flags que controlan qué parámetros se optimizan y cuáles se fijan.

Nota:
tanto los parámetros de cámara como los coeficientes de distorsión, pueden venir de calibraciones previas, en ese caso se toman como punto de partida para mejorarlas iterarivamente.

**devuelve**

- `R`: Matriz de rotación que transforma puntos de la cámara 1 a la cámara 2.
- `T`: Vector de traslación entre cámaras.
- `E`: Matriz esencial.
- `F`: Matriz fundamental.
  
Además puede devolver:
- Pose de cámara para cada imagen (`rvecs`, `tvecs`), y su correspondiente estimación de confianza.



In [ ]:
# Calibramos Stereo

print("Calibrating Stereo")
print("num images:", len(left_images_points))

err, left_K, left_dist, right_K, right_dist, R, T, E, F = cv2.stereoCalibrate(
    world_points, 
    left_images_points, 
    right_images_points, 
    None, 
    None, 
    None, 
    None, 
    image_size, 
    flags=0
)

In [ ]:
from calib import np_print

to_print = [

    "# Left camera Intrinsics:",
    ("left_K", left_K),
    ("left_dist", left_dist),

    "# Right camera Intrinsics:",
    ("right_K", right_K),
    ("right_dist", right_dist),

    "# Rotation:",
    ("R", R),

    "# Translation:",
    ("T", T),
    
    "# Essential Matrix:",
    ("E", E),
    
    "# Fundamental Matrix:",
    ("F", F),
        
]

In [ ]:
print("# STEREO CALIBRATION")
for line in to_print:

    if isinstance(line, str):   
        print(line)
    else:
        var_name, np_array = line
        print(f"{var_name} = {np_print(np_array)}\n")

In [ ]:
# serializamos los resultados en un pickle
import pickle

calibration_results = {
    'left_K': left_K,
    'left_dist': left_dist,
    'right_K': right_K,
    'right_dist': right_dist,
    'R': R,
    'T': T,
    'E': E,
    'F': F,
    'image_size': image_size,
}

calibration_file = os.path.join("stereo_data", "stereo_calibration.pkl")
with open(calibration_file, "wb") as f:
    f.write(pickle.dumps(calibration_results))
        


In [ ]:
# para recuperar los resultados podemos cargarlos usando pickle:

with open(calibration_file, "rb") as f:
    results2 = pickle.loads(f.read())

In [ ]:
results2.keys()

# Rectificación Estéreo

La rectificación estéreo es el proceso de transformar dos imágenes capturadas por cámaras estéreo para que sus planos de imagen queden coplanares, es decir, que las líneas epipolares se vuelvan horizontales y alineadas entre ambas imágenes. Esto facilita la búsqueda de correspondencias, ya que cualquier punto en una imagen tendrá su correspondiente en la otra imagen a lo largo de la misma línea horizontal.

## Rectificación Estéreo en OpenCV

*"básicamente encuentra transformaciones tal que alinean las imágenes en el mismo plano"*

La función [`cv2.stereoRectify`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga617b1685d4059c6040827800e72ad2b6) de OpenCV, a partir de una calibración estéreo previa (que por ejemplo podemos obtener usando stereoCalibrate), calcula matrices necesarias para rectificar ambas cámaras, es decir, transformar las imágenes de las cámaras de modo que:

- los planos de imagen sean coplanares
- las líneas epipolares se alineen (horizontalmente o verticalmente)
- se facilite el cálculo de correspondencias estéreo

**parámetros de entrada**
- `cameraMatrix1`,  `distCoeffs1`: matriz intrínseca camara 1 (izquierda), y sus coeficientes de distorsión.
- `cameraMatrix2`, `distCoeffs2`: matriz intrínseca camara 2 (derecha), y sus coeficientes de distorsión.
- `R`, `T`: pose relativa entre cámara 1 y camara 2, (tal como la que se devuelve en stereoCalibrate)

Otros parámetros de entrada:
- `flags`: si se usa el flag `CALIB_ZERO_DISPARITY`, los puntos principales se alinean en el sistema rectificado.
- `alpha`: un float entre 0 y 1, o -1. 0 indica que no se desean dejar píxeles vacíos en las imágenes resultantes, recortará más imagen. 1 indica que no se debe descartar ningún pixel. -1 deja a OpenCV elegir un valor intermedio de manera automática.
- `newImageSize`: Permite especificar una nueva resolución de salida para las imágenes rectificadas.

**devuelve**

- `R1`, `R2`: matrices de rotación de rectificación para cada cámara. Estas matrices llevan puntos desde el sistema de coordenadas no rectificado al rectificado, haciendo que ambos planos de imagen queden alineados.

- `P1`, `P2`
Son las matrices de proyección para cada cámara en el nuevo sistema rectificado.

    Es decir, 
$P_1 = K' \begin{bmatrix} I & 0 \end{bmatrix}$, $P_2 = K' \begin{bmatrix} I & t' \end{bmatrix}$, en donde:

    - K' es la nueva matriz intrínseca común (derivada a partir de las intrínsecas originales y los parámetros alpha, newImageSize)
    - t' es un desplazamiento a lo largo del eje x (en estéreo horizontal) o del eje y (en estéreo vertical)



- `Q` Es la matriz de reproyección. Permite convertir un punto (x, y, disparity) en un punto en 3D.

- `validPixROI1`, `validPixROI2`
Son los regiones de interés (rectángulos) donde los píxeles rectificados son válidos (sin relleno negro). Útiles para recortar las imágenes y evitar bordes no válidos después de la rectificación.
  

In [ ]:
# realicemos la rectificación estéreo:

R1, R2, P1, P2, Q, validRoi1, validRoi2 = cv2.stereoRectify(
    left_K, left_dist, right_K, right_dist, image_size, R, T, alpha=0
)


### Mapas de rectificación Stereo en OpenCV

En este caso usaremos función [`cv2.initUndistortRectifyMap`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga7dfb72c9cf9780a347fbe3d1c47e5d5a) para realizar la des-distorsión y rectificación conjunta.

*"básicamente, precomputa resultados intermedios para rectificar las imágenes píxel a píxel, de manera eficiente"*

El resultado queda expresado en dos matrices (o mapas de rectificación map1 y map2), que se utilizan con la función `cv2.remap`.


Las imagenes rectificadas resultantes serán las originales, pero transformadas de forma tal que:

- hubieran sido capturadas por otra cámara con matriz `newCameraMatrix` y distorsión cero.
- hubieran estado rotadas apuntando en la orientación `R`


**nueva matriz de cámara**

Para el caso monocular, `newCameraMatrix` se puede calcular usando `getOptimalNewCameraMatrix`, que controla la escala recortando partes de imagen sin píxeles. Para el caso de cámara stereo, `newCameraMatrix` se usa con las matrices de proyección `P1` o `P2` computadas por `stereoRectify`.

**nueva orientación R**
Para estéreo podemos alinear cada cámara individual, usando `R1` o `R2` encontradas en la rectificación estéreo.
De esa manera las líneas epipolares de ambas imágenes quedarán horizontales y tendrán la misma coordenada $y$.

**parámetros de entrada**
- `cameraMatrix`, `distCoeffs` matriz de cámara original con los parámetros intrínsecos (`K`), y los coeficientes de distorsión. Los coeficinetes pueden ser nulos, y se asume cero distorsión.
- `R` matriz de rotación de 3x3. Podemos usar `R1` o `R2`, que computamos usando `stereoRectify`. Si se omite, entonces se asume la identidad.
- `newCameraMatrix`, los parámetros de cámara destino.
- `size`, el tamaño de la imagen destino.
- `m1type`, tipo de dato a usar. Por ejemplo, si queremos real-time podemos usar `CV_16SC2`, si no podemos usar `CV_32FC1` que generará salidas en *float32* en 1 canal.

**salida**
- `map1`, `map2`, (mapa x, y mapa y) los mapas de des-distorsión-rectificación.

In [ ]:
# calculemos los mapas de des-distorsión-rectificación:

left_map_x, left_map_y = cv2.initUndistortRectifyMap(left_K, left_dist, R1, P1, image_size, cv2.CV_32FC1)
right_map_x, right_map_y = cv2.initUndistortRectifyMap(right_K, right_dist, R2, P2, image_size, cv2.CV_32FC1)

In [ ]:
# Qué pinta tienen estos mapas?
# chequeemos el shape, debería coincidir con image_size
left_map_x.shape

In [ ]:
# Qué valores contiene?
# en este caso son floats, por lo que se necesitará alguna forma de interpolación
left_map_x[0:3, 0:3]

### remap
La función de OpenCV [`cv2.remap`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#gab75ef31ce5cdfb5c44b6da5f3b908ea4) permite aplicar los mapas de rectificación de manera eficiente.

Si los mapas son floats de un solo canal, entonces:

- $x_{map} = map_x(x_{dest}, y_{dest})$
- $y_{map} = map_y(x_{dest}, y_{dest})$

para obtener la posición en la imagen original, como $x_{map}$ y $y_{map}$ son floats, tenemos que interpolar y quedarnos con valores que buscaremos en posiciones enteras.

La opción más fácil es redondear las posiciones al vecino más cercano `INTER_NEAREST`.

- $x_{ori} = round(x_{map})$
- $y_{ori} = round(y_{map})$

Y luego la función asigna:

$I_{rect}(x_{dest}, y_{dest}) := I_{ori}(x_{ori}, y_{ori})$

También podemos pasar el flag `INTER_LINEAR`, con la que realizará una interpolación bilinear en base a los 4 vecinos más cercanos.




In [ ]:
# tomemos un par estéreo
image_index = 22

left_image = left_images[image_index]
right_image = right_images[image_index]

In [ ]:
# muestro las imagenes:
show_images([
    left_image, right_image
], [
    "left original", "right original"
])


In [ ]:
# aplicamos des-distorsion + rectificación, con los mapas que calculamos ateriormente
left_image_rectified = cv2.remap(left_image, left_map_x, left_map_y, cv2.INTER_LINEAR)
right_image_rectified = cv2.remap(right_image, right_map_x, right_map_y, cv2.INTER_LINEAR)

In [ ]:
import numpy as np

# Ejemplo, para saber qué valor poner en la posición (i,j) de la imagen destino,
# a cuál píxel en la imagen original tendríamos que ir buscar?
x_dest, y_dest = 321, 123

x_ori = left_map_x[y_dest, x_dest]
y_ori = left_map_y[y_dest, x_dest]

x_ori = int(np.round(x_ori))
y_ori = int(np.round(y_ori))

print(f"using nearest neighbor interpolation, I_dest[{y_dest}, {x_dest}] = I_ori[{y_ori}, {x_ori}]")

print(left_image[y_ori, x_ori])
print(left_image_rectified[y_dest, x_dest])


In [ ]:
# muestro las imagenes stereo-rectificadas
# observar el vertice de la caja, de la notebook, y del cuaderno. 
# coinciden perfectamente en y.
from matplotlib import pyplot as plt 

fig, axes = show_images([
    left_image_rectified, right_image_rectified
], [
    "left rectified", "right rectified"
], show=False)

for y, c in zip([160, 533, 900], ['r', 'b', 'c']):
    axes[0].axhline(y=y, color=c, linestyle='-', linewidth=0.5)
    axes[1].axhline(y=y, color=c, linestyle='-', linewidth=0.5)
plt.show()

In [ ]:
# Guardemos los resultados de rectificación para usar posteriormente:

stereo_maps = {

    # undistorting maps
    "left_map_x": left_map_x,
    "left_map_y": left_map_y,
    "right_map_x": right_map_x,
    "right_map_y": right_map_y,

    # add also rectifying info:
    "R1": R1,
    "R2": R2,
    "P1": P1,
    "P2": P2,
    "Q": Q,
    "validRoi1": validRoi1,
    "validRoi2": validRoi2,

}

stereo_maps_file = os.path.join("stereo_data", "stereo_maps.pkl")
with open(stereo_maps_file, "wb") as f:
    f.write(pickle.dumps(stereo_maps))
        

# Tool de Calibración Stereo

La propuesta es utilizar la
[herramienta de calibración stereo](https://github.com/udesa-vision/i308-calib) que integra todo el código explicado anteriormente.

**NOTA: esta herramienta utiliza la GUI de ventanas del sistema operativo No funcionará en Google Colab**.


Necesitaremos un checkerboard físico con el que calibraremos. **Importante** asegurarse de que está pegado en una base rígida para evitar que se pandee.


## Instalación:

Se puede instalar via pip:

```bash

    pip install -qq git+https://github.com/udesa-vision/i308-calib.git

```

## Configurar herramienta

Ejecutar el comando que inicia la herramienta de calibración stereo:

```bash

    calib-stereo -v 0 -r 3840x1080

```

### Argumentos

- **video** str or int. 
Especifica el dispositivo de video, en linux podría ser algo del estilo `/dev/video<N>`

- **resolution** str (optional)
 la resolución solicitada en el formato "`<width>`x`<height>`" en pixels

- **checkerboard** str (optional) default=10x7
 distribución del checkerboard "`<width>`x`<height>`" en cuadrados

- **square-size** str (optional) default=24.2
 el tamaño de cuadrado en milímetros

- **config** str (optional)
un archivo .yaml de configuración de captura

- **data** str (optional) default=data/stereo
directorio donde se guardará el dataset de calibración, capturas y los resultados de calibración




### Archivo de configuración de captura

Se proporciona el archivo de configuracion `stereo.yaml`

Ejecuta el comando:


```bash

 copy-configs

```

Esto debería copiar el archivo `cfg/stereo.yaml`.


Ejemplo de archivo de configuración de captura (.yaml):

```yaml

    video: 0
    name: Stereo ELP-USB3D1080P02
    resolution: 3840x1080

    resolutions:
        - 3840x1080
        - 2560x720
        - 1600x600
        - 1280x480
        - 640x240
    
    # capture_mode: 'dshow'
    # compression: 'MJPG'
    # threaded: False

```

Podemos editar el archivo o crear uno nuevo para configurar nuestro dispositivo.


### Sobreescribiendo parámetros de captura

En caso de utilizar archivo de configuración, algunos parametros como `video` y `resolution` se pueden sobreescribir, con los argumentos de la linea de comandos:

Por ejemplo, el siguiente comando va a utilizar el video 3, independientemente de lo que esté configurado en el archivo .yaml:

```bash

    calib-stereo --video 3 --config cfg/stereo.yaml 

```


## Usando herramienta de calibración Stereo


Ejecutar la herramienta, usando los parámetros correspondientes:

```bash

calib-stereo --video 0 --resolution 3840x1080 --config cfg/stereo.yaml --checkerboard 10x7 --square-size 24.2

```

deberíamos ver la captura stereo:

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_06/calib-stereo/tool01.jpg?raw=true" />


---

Presionamos la tecla `d` para habilitar la detección, luego vamos a gregando elementos al dataset de calibración stereo con la tecla `a`:

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_06/calib-stereo/tool02.jpg?raw=true" />


---

Agregamos ejemplos hasta cubrir todas las regiones de los sensores de las cámaras.
Es importante variar lo más posible las posiciones y la orientación del checkerboard (ej, rotado 90º, más cerca, más lejos, en diagonal y con gran variación en el eje Z), esto dará mayor estabilidad al algoritmo de calibración.
En especial, los corners de la imagen, son los lugares de mayor distorsión y es conveniente tener ejemplos que cubran esas áreas.

Capturamos al menos 10 imagenes, idealmente más de 20.

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_06/calib-stereo/tool03.jpg?raw=true" />


---

Una vez que tenemos una buena cantidad de ejemplos presionamos la tecla `c` para realizar la calibración stereo.
Y a continuación la tecla `m` para computar los mapas de rectificación stereo.


Deberíamos obtener un output similar a este:

```python

# STEREO CALIBRATION
# Left camera Intrinsics:
left_K = np.array([
	[   602.068,	     0.000,	   956.396],
	[     0.000,	   602.654,	   551.320],
	[     0.000,	     0.000,	     1.000]
])

left_dist = np.array([
	[  0.010065,	 -0.028070,	  0.000550,	  0.000168,	  0.004580]
])

# Right camera Intrinsics:
right_K = np.array([
	[   598.799,	     0.000,	   957.412],
	[     0.000,	   599.406,	   536.519],
	[     0.000,	     0.000,	     1.000]
])

right_dist = np.array([
	[  0.012286,	 -0.030444,	  0.000265,	 -0.000065,	  0.005175]
])

# Rotation:
R = np.array([
	[     1.000,	    -0.000,	     0.005],
	[     0.000,	     1.000,	     0.008],
	[    -0.005,	    -0.008,	     1.000]
])

# Translation:
T = np.array([
	[-59.868688],
	[  0.628902],
	[ -0.500083]
])

# Essential Matrix:
E = np.array([
	[    -0.003,	     0.495,	     0.633],
	[    -0.809,	    -0.483,	    59.863],
	[    -0.652,	   -59.866,	    -0.486]
])

# Fundamental Matrix:
F = np.array([
	[     0.000,	    -0.000,	    -0.003],
	[     0.000,	     0.000,	    -0.962],
	[    -0.001,	     0.935,	     1.000]
])

```

Estos datos se serializan como pickle en el directorio de datos.


## Notas de uso
- presionando `q` terminamos el programa
- presionando `h` vemos el menú de ayuda con las teclas y sus correspondientes comandos.
- si algo sale catastróficamente mal simplemente borrar el directorio `data` y comenzar de nuevo.
- si la tool se cerró y queremos agregar nuevos ejemplos al dataset de calibración podemos cargar las imágenes del dataset con la tecla `l`
- Cuando el algoritmo no ve el tablero, tarda más que cuando lo ve. en caso de andar lento se puede apagar la deteccion con la letra `d` centrar el tablero y volver a activar la detección.
- presionando `s` podemos tomar capturas de objetos, que se guardarán en el directorio `data/captures`

## Rectificación Stereo en Runtime

In [ ]:
import os

left_image_file = os.path.join("stereo_data", "captures", "capture_left_10.jpg")
right_image_file = os.path.join("stereo_data", "captures", "capture_right_10.jpg")

left_image = cv2.imread(left_image_file, 0)
right_image = cv2.imread(right_image_file, 0)


In [ ]:

fig, axes = show_images([
    left_image, right_image
], [
    "left original", "right original"
], show=False)



In [ ]:
# cargamos la información de rectificación stereo, que generamos offline

import pickle


stereo_calib_file = os.path.join("stereo_data", "stereo_calibration.pkl")
stereo_maps_file = os.path.join("stereo_data", "stereo_maps.pkl")

print("reading stereo calibration results...")
with open(stereo_calib_file, "rb") as f:
    calibration = pickle.loads(f.read())
    
print("reading stereo rectification maps...")
with open(stereo_maps_file, "rb") as f:
    maps = pickle.loads(f.read())

In [ ]:
left_map_x, left_map_y = maps['left_map_x'], maps['left_map_y']
right_map_x, right_map_y = maps['right_map_x'], maps['right_map_y']

In [ ]:

left_image_rectified = cv2.remap(left_image, left_map_x, left_map_y, cv2.INTER_LINEAR)
right_image_rectified = cv2.remap(right_image, right_map_x, right_map_y, cv2.INTER_LINEAR)

cv2.imwrite("cat_left.jpg", left_image_rectified)
cv2.imwrite("cat_right.jpg", right_image_rectified)


fig, axes = show_images([
    left_image_rectified, right_image_rectified
], [
    "left rectified", "right rectified"
], show=False)

for y, c in zip([255, 470, 830], ['r', 'b', 'c']):
    axes[0].axhline(y=y, color=c, linestyle='-', linewidth=0.5)
    axes[1].axhline(y=y, color=c, linestyle='-', linewidth=0.5)
plt.show()

# Disparidad

Ahora que ya hicimos la calibración estéreo, podemos rectificar el par de imágenes estéreo de modo tal que las correspondencias izquierdas y derechas caigan perfectamente en la misma línea.

La disparidad es simplemente **la diferencia horizontal de dos puntos correspondientes (en la misma coordenada y)**

¿cómo hacemos para encontrar las correspondencias?

Podríamos intentar comparar los valores de gris de los píxeles en la misma fila del par de imágenes estéreo. 

Este método no es muy robusto. Por ejemplo, píxeles que no son correspondientes en las diferentes imágenes pueden tener la misma intensidad. Por otro lado,
para correspondencias buenas, los valores de gris en las cámaras distintas puede no coincidir.


## Block Matching

El algoritmo de Block Matching es uno de los métodos más comunes para calcular disparidad dada un par de imágenes estéreo rectificado.

¿Cómo funciona Block Matching?

1. Búsqueda de bloques:
El algoritmo se basa en dividir la imagen en pequeñas ventanas o bloques de píxeles. Se toma un bloque centrado en un píxel de la imagen izquierda y se busca el bloque correspondiente en la imagen derecha. El objetivo es encontrar el bloque en la imagen derecha que sea lo más similar posible al bloque de la imagen izquierda. La diferencia en la posición horizontal entre los bloques es la disparidad.

2. Ventana de búsqueda (disparidad máxima):
La búsqueda no se realiza en toda la imagen derecha, sino dentro de una ventana de búsqueda horizontal limitada. Esto se debe a que los objetos cercanos tienen grandes diferencias de posición entre las dos imágenes, mientras que los objetos lejanos tienen pequeñas diferencias. Por lo tanto, solo se busca dentro de un rango limitado de posiciones de disparidad, lo que reduce el tiempo de cálculo y el riesgo de errores.

3. Medición de similitud:
Para decidir cuál es el bloque de la imagen derecha más similar al bloque de la imagen izquierda, el algoritmo utiliza una función de similitud o costo. Las funciones de similitud más comunes son:

- SSD (Sum of Squared Differences): Suma de las diferencias cuadradas de los valores de píxeles entre los bloques.

	$ SSD = \sum_{i,j} (I_{izq}(x+i, y+j) - I_{der}(x+i-d, y+j))^2 $

	en donde d es la disparidad.

- SAD (Sum of Absolute Differences): Suma de las diferencias absolutas entre los valores de los píxeles en los bloques.
- NCC (Normalized Cross-Correlation): Otra métrica más robusta que mide la correlación entre bloques.


4. Selección de la disparidad:
Para calcular la disparidad, el algoritmo prueba múltiples valores de $d$ y calcula la SSD (o la métrica elegida) para cada uno. La disparidad para un píxel se determina eligiendo el valor de d que minimiza la SSD (es decir, donde los bloques de las dos imágenes coinciden mejor).
 
6. Repetición para cada píxel:
El proceso se repite para cada píxel de la imagen izquierda, generando un mapa de disparidad denso, que contiene la disparidad calculada para cada píxel de la imagen. Se puede utilizar programación dinámica o imágenes integrales para acelerar este paso.


## Block Matching con OpenCV

OpenCV define una familia de algoritmos de disparidad [StereoMatcher](https://docs.opencv.org/4.10.0/d2/d6e/classcv_1_1StereoMatcher.html).
La implementación de [BlockMatching](https://docs.opencv.org/4.10.0/d9/dba/classcv_1_1StereoBM.html) extiende dicha interface base.

### Parámetros de Block Matching:

- **numDisparities:** es el número máximo de disparidades (desplazamientos) a probar. Debe ser un múltiplo de 16. Valores mayores permiten detectar objetos a mayores distancias (más lejos de las cámaras), pero incrementa el tiempo de cálculo. Si nos interesan objetos más cercanos, podemos probar con valores menores.

- **blockSize:** Define el tamaño del bloque de píxeles que el algoritmo utiliza para comparar las dos imágenes. Valores mayores hace que las áreas homogéneas (con pocas texturas) se procesen mejor, pero puede perder detalles finos y aumenta el tiempo de cálculo. Valores menores funcionan mejor si queremos más detalles, pero nos introduce más ruido.

- **preFilterSize:** Define el tamaño del filtro que se aplica antes de la comparación de bloques para reducir el ruido en las imágenes. El valor debe ser impar y estar entre 5 y 255. Valores mas grandes reducen más el ruido, pero pueden eliminar detalles. Menores valores mantienen más detalles, pero es más sensible al ruido.

- **preFilterCap:** Limita (clipea) los valores filtrados para mejorar la robustez del algoritmo. Los valores de píxel filtrados se limitan entre -preFilterCap y preFilterCap. Entre 1 y 31.

- **minDisparity:** Establece el valor mínimo de disparidad. En la mayoría de los casos, se usa 0, pero si se sabe que algunos objetos en la escena tienen disparidades negativas, se puede ajustar a un valor menor.

- **textureThreshold:** Se utiliza para ignorar áreas con muy poca textura. Si el contraste dentro de un bloque es menor que este umbral, el algoritmo asigna una disparidad de 0 (o ignora el bloque). Un valor mayor mejora la precisión en áreas con baja textura. Mayor valor, evita errores en áreas homogéneas, pero puede ignorar pequeños detalles.

- **uniquenessRatio:** Controla cuán único debe ser el bloque coincidente. Esto es útil para eliminar errores ambiguos en la correspondencia. Mayor valor, hace que la coincidencia sea más estricta, evitando falsas correspondencias, pero podría ignorar algunos puntos de la imagen. Menor valor, acepta más correspondencias, pero podría aumentar el ruido.

- **speckleWindowSize:** Elimina pequeñas regiones aisladas en el mapa de disparidad que probablemente son ruido. Define el tamaño máximo de una región de píxeles conectados que pueden ser considerados como ruido y eliminados. Mayor valor, Filtra más ruido, pero podría eliminar detalles finos. Menor valor, mantiene más detalles, pero podría dejar más ruido.

- **speckleRange:** Define la diferencia máxima de disparidad entre píxeles conectados dentro de una región. Esto se usa junto con speckleWindowSize para determinar si una región debe ser eliminada.

- **disp12MaxDiff:** Controla la diferencia máxima permitida entre las disparidades calculadas para las imágenes izquierda y derecha (consistencia izquierda-derecha). Si la diferencia entre las disparidades excede este valor, se descarta la disparidad. Mayor valor, acepta más disparidades, pero puede introducir errores. Menor valor, hace que el algoritmo sea más estricto, eliminando más errores, pero podría perder algunos píxeles válidos.



In [ ]:
import numpy as np

stereo = cv2.StereoBM_create(
    numDisparities=256,
    blockSize=15,
)

stereo.setPreFilterSize(31)
stereo.setPreFilterCap(15)
stereo.setMinDisparity(0)
stereo.setTextureThreshold(7)
stereo.setUniquenessRatio(3)
stereo.setSpeckleWindowSize(512)
# stereo.setSpeckleWindowSize(0)
stereo.setSpeckleRange(32)
stereo.setDisp12MaxDiff(23)


# Cálculo del mapa de disparidad
disparity_map = stereo.compute(
    left_image_rectified, 
    right_image_rectified
).astype(np.float32) / 16.0

plt.figure(figsize=(12, 6))
plt.imshow(disparity_map)
plt.colorbar()
plt.show()


## Semi-Global Block Matching SGBM

StereoSGBM (Semi-Global Block Matching) es una variante mejorada del algoritmo de Block Matching tradicional (StereoBM). 
Pero es computacionalmente más caro.

Aunque ambos algoritmos calculan un mapa de disparidad entre imágenes estéreo, StereoSGBM introduce un enfoque de correspondencia semi-global, que mejora la precisión en áreas con poca textura y bordes suaves, donde el enfoque clásico de bloques de StereoBM falla. 

El enfoque semi-global, optimiza la disparidad no solo a nivel local, sino también considerando múltiples direcciones alrededor de cada píxel, lo que mejora la coherencia del mapa de disparidad.

1. Búsqueda de bloques: Es similar a BM, pero en lugar de tomar la mejor correspondencia inmediatamente, StereoSGBM sigue optimizando la solución considerando también la información global, asegurando la coherencia en la disparidad a lo largo de la imagen.
   
2. Función de costo, combina las métricas anteriores con una penalización de cambios bruscos de disparidad entre píxeles vecinos (costo suavizado).

### Parámetros adicionales:

- P1 y P2: Estos parámetros controlan la suavidad de las disparidades.  

  - P1: penaliza cambios pequeños de disparidad entre píxeles vecinos. Un valor mayor suaviza la transición entre píxeles.
  - P2: penaliza cambios bruscos de disparidad entre píxeles vecinos. Un valor mayor evita que las disparidades varíen mucho entre píxeles cercanos.

Generalmente, P2 debe ser mayor que P1.


In [ ]:
import numpy as np

stereo = cv2.StereoSGBM_create(
    numDisparities=128,
    blockSize=7
)

# stereo.setBlockSize(3)
# stereo.setMode()

stereo.setP1(16)
stereo.setP2(128)
stereo.setUniquenessRatio(7)
stereo.setSpeckleWindowSize(512)
stereo.setSpeckleRange(64)
stereo.setDisp12MaxDiff(1)
stereo.setPreFilterCap(63)


# Cálculo del mapa de disparidad
disparity_map = stereo.compute(
    left_image_rectified, 
    right_image_rectified
).astype(np.float32) / 16.0

plt.figure(figsize=(12, 6))
plt.imshow(disparity_map)
plt.colorbar()
plt.show()

In [ ]:
!pip install -qq onnxruntime

In [ ]:
w, h = left_image.shape[1], left_image.shape[0]

fx = left_K[0][0]
fy = left_K[1][1]
cx0 = left_K[0][2]
cy0 = left_K[1][2]

baseline = np.linalg.norm(T)

In [ ]:
# Cre Stereo

import cv2
import json
import numpy as np
from pathlib import Path

from disparity.method_cre_stereo import CREStereo
from disparity.method_opencv_bm import StereoBM, StereoSGBM
from disparity.methods import Calibration, InputPair, Config


calibration = Calibration(**{
    "width": w,
    "height": h,
    "baseline_meters": baseline / 1000,
    "fx": fx,
    "fy": fy,
    "cx0": cx0,
    "cx1": cx0,
    "cy": cy0,
    "depth_range": [0.05, 20.0],
    "left_image_rect_normalized": [0, 0, 1, 1]
})



In [ ]:
import os

models_path = "models"
if not os.path.exists(models_path):
    os.makedirs(models_path)

In [ ]:
#models_path = Path.home() / ".cache" / "stereodemo" / "models"
models_path = Path(models_path)
pair = InputPair(left_image_rectified, right_image_rectified, calibration)
# pair = InputPair(left_image, right_image, calibration)
config = Config(models_path=models_path)

# params = {
#    "Shape": "1280x720",
#    "Mode": "combined",
#    "Iterations": 20
#}
method = CREStereo(config)

#method.parameters["Shape"].set_value("640x480")
method.parameters["Shape"].set_value("1280x720")
# method.parameters["Iterations"].set_value("10")

#method.parameters.update(params)
# method = StereoBM(config)
# method = StereoSGBM(config)
# method = StereoBM(config)
disparity = method.compute_disparity(pair)

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(disparity.disparity_pixels)
plt.colorbar()
plt.show()

# Cálculo de profundidad

Una vez que tenemos listo el mapa de disparidad, podemos calcular el mapa de profundidad, usando la siguiente fórmula:

$ z = \frac{f . B}{d} $


In [ ]:
def compute_depth(disparity_map, f, B, default=1000.0):

    # Crea una copia del mapa de disparidad
    disparity_map = disparity_map.copy()
    
    # Evita divisiones por cero o disparidades negativas (les asignamos el valor default)
    mask_invalid = (disparity_map <= 0)
    
    # Calcula la profundidad con la fórmula Z = f * B / disparidad
    depth_map = np.zeros_like(disparity_map, dtype=np.float32)
    depth_map[~mask_invalid] = (f * B) / disparity_map[~mask_invalid]
    
    # Asigna valor fijo a los puntos donde la disparidad es inválida
    depth_map[mask_invalid] = default
    
    return depth_map

In [ ]:
# disparity = np.load("disparity.npz")["arr_0"]
disparity_map = disparity.disparity_pixels

In [ ]:
f = left_K[0][0]
B = baseline = np.linalg.norm(T) 

depth = compute_depth(disparity_map, f, B)

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(depth)
plt.colorbar()
plt.show()

In [ ]:
depth2 = depth.copy()
depth2[depth2 > 10000] = 10000 

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(depth2)
plt.colorbar()
plt.show()

In [ ]:
depth2 = depth.copy()
depth2[depth2 > 10000] = 10000 

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(depth2)
plt.colorbar()
plt.show()

# Exportando a PLY

El formato .PLY (Polygon File Format) también llamado "Stanford Triangle Format", se utiliza para almacenar mallas 3D con vértices, caras y opcionalmente, propiedades como colores o normales.

Podemos visualizar este formato de archivos usando la herramienta [meshlab](https://www.meshlab.net/#download)

In [ ]:
def export_ply(archivo_ply, depth_map, img_color, f, cx, cy):
    """
    Exporta el mapa de profundidad y la imagen de color a un archivo .PLY.

    Parámetros:
        archivo_ply (str): Ruta del archivo de salida .PLY.
        depth_map (numpy.ndarray): Mapa de profundidad.
        img_color (numpy.ndarray): Imagen original en color.
        f (float): Distancia focal de la cámara (en píxeles).
        cx, cy (float): Coordenadas del centro óptico (en píxeles).
    """
    h, w = depth_map.shape
    with open(archivo_ply, 'w') as f_out:
        # Cabecera del archivo .PLY
        f_out.write('ply\n')
        f_out.write('format ascii 1.0\n')
        f_out.write(f'element vertex {h * w}\n')
        f_out.write('property float x\n')
        f_out.write('property float y\n')
        f_out.write('property float z\n')
        f_out.write('property uchar red\n')
        f_out.write('property uchar green\n')
        f_out.write('property uchar blue\n')
        f_out.write('end_header\n')

        for y in range(h):
            for x in range(w):
                Z = depth_map[y, x]
                if Z == 0:  # skippea píxeles con profundidad 0
                    continue
                
                # convierte coordenadas de píxel a coordenadas de mundo 3D
                X = (x - cx) * Z / f
                Y = (y - cy) * Z / f

                # color de la imagen original (en BGR)
                color = img_color[y, x]
                r, g, b = color[2], color[1], color[0]

                # escribie el punto 3D y su color al archivo PLY
                f_out.write(f'{X} {Y} {Z} {r} {g} {b}\n')

    print(f'Archivo .PLY guardado como {archivo_ply}')

In [ ]:
left_image_color = cv2.imread(left_image_file)

In [ ]:
left_image_color_rectified = cv2.remap(left_image_color, left_map_x, left_map_y, cv2.INTER_LINEAR)


In [ ]:

export_ply(
    "gato.ply",
    depth2,
    left_image_color_rectified,
    left_K[0, 0],
    left_K[0, 2],
    left_K[1, 2],

)